# Qwen chatbot with file RAG (Chroma) and web search on Colab
Chat model **Qwen2.5-Coder-7B-Instruct** (4-bit) with two tools: file retrieval and web search.

- **File RAG:** upload PDF/TXT, split with LangChain, embed with `all-MiniLM-L6-v2`, store in **Chroma**, answer from the top matches.
- **Web search:** DuckDuckGo, switched on per request.
- **Serving:** FastAPI endpoint (public through ngrok) and a Gradio UI.

Paste your own ngrok token where marked `PASTE_YOUR_NGROK_TOKEN_HERE`.


In [2]:
# 1. Sab kuch uninstall karein (Safayi)
!pip uninstall -y torch torchvision torchaudio transformers bitsandbytes accelerate

# 2. Stable PyTorch Install karein (Ye version har cheez ke sath chalta hai)
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 3. Baqi Libraries Install karein
!pip install transformers==4.46.1 bitsandbytes==0.44.1 accelerate==1.1.1 fastapi uvicorn pyngrok nest_asyncio gradio pydantic langchain langchain-community langchain-text-splitters pypdf sentence-transformers chromadb duckduckgo-search

print("✅ Repair Complete! Ab Step 2 zaroor karein.")
!pip install -U ddgs
!pip install -U duckduckgo-search

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading
import gradio as gr
import os

# RAG & Search Imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.tools import DuckDuckGoSearchRun

# ==========================================
# 🔑 APNA NGROK TOKEN YAHAN PASTE KAREIN
# ==========================================
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
# ==========================================

if NGROK_AUTH_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
else:
    print("⚠️ WARNING: Ngrok Token missing! API Public nahi hogi.")

# ==========================================
# 🚀 LOAD MODEL (The Brain)
# ==========================================
print("🧠 Model Load ho raha hai... (PyTorch 2.5.1 Stable Mode)")
model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Qwen-Coder Online!")

# ==========================================
# 🚀 TOOLS (RAG + WEB SEARCH)
# ==========================================
vector_db = None
search_tool = DuckDuckGoSearchRun()

def process_files_for_rag(files):
    global vector_db
    if not files: return "⚠️ Koi file nahi mili."

    documents = []
    status = ""
    for file in files:
        try:
            if file.name.endswith(".pdf"):
                loader = PyPDFLoader(file.name)
                documents.extend(loader.load())
                status += f"📄 PDF: {os.path.basename(file.name)}\n"
            elif file.name.endswith(".txt"):
                loader = TextLoader(file.name)
                documents.extend(loader.load())
                status += f"📝 TXT: {os.path.basename(file.name)}\n"
        except Exception as e:
            status += f"❌ Error: {file.name} - {str(e)}\n"

    if documents:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = text_splitter.split_documents(documents)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vector_db = Chroma.from_documents(chunks, embeddings)
        return status + f"\n✅ Database Ready! ({len(chunks)} chunks)"
    return "❌ Koi text extract nahi hua."

# ==========================================
# 🚀 SMART CHAT LOGIC
# ==========================================
def smart_chat(prompt, history, use_web=False):
    global vector_db

    context_text = ""
    system_instruction = "You are an expert Developer & AI Assistant. Answer in Roman Urdu mixed with English."

    # 1. RAG Check
    if vector_db:
        docs = vector_db.similarity_search(prompt, k=3)
        rag_content = "\n".join([d.page_content for d in docs])
        context_text += f"\n\n[📂 File Context]:\n{rag_content}"
        system_instruction += " Use the File Context to answer."

    # 2. Web Search Check
    if use_web:
        try:
            print("🌐 Searching Internet...")
            web_results = search_tool.run(prompt)
            context_text += f"\n\n[🌍 Web Search Results]:\n{web_results}"
            system_instruction += " Use the Web Search Results for latest info."
        except Exception as e:
            context_text += f"\n[Web Error]: {str(e)}"

    # 3. Prompt Construction
    messages = [{"role": "system", "content": system_instruction}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    final_user_message = f"{prompt}\n{context_text}"
    messages.append({"role": "user", "content": final_user_message})

    # 4. Generate
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# ==========================================
# 🚀 API SETUP (FastAPI)
# ==========================================
app = FastAPI()

class APIRequest(BaseModel):
    prompt: str
    use_web: bool = False

@app.post("/chat")
def api_endpoint(req: APIRequest):
    response = smart_chat(req.prompt, [], req.use_web)
    return {"response": response}

def run_api():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

try:
    public_url = ngrok.connect(8000).public_url
    print(f"\n🌍 API LIVE: {public_url}")
except:
    print("⚠️ Ngrok Error. Token check karein.")

# ==========================================
# 🚀 UI SETUP (Gradio)
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(), title="Ultimate AI Studio") as demo:
    gr.Markdown("# 🚀 Ultimate AI Studio (Chat + Files + Internet + API)")
    gr.Markdown(f"**API Endpoint:** `{public_url}/chat`")

    with gr.Row():
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 📂 Upload Knowledge")
            files = gr.File(file_count="multiple", label="PDF/TXT")
            upload_btn = gr.Button("Process Files", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)

            gr.Markdown("### 🌐 Tools")
            web_checkbox = gr.Checkbox(label="Enable Internet Search", value=False)

        with gr.Column(scale=3):
            chat = gr.ChatInterface(
                fn=smart_chat,
                additional_inputs=[web_checkbox],
                title="Qwen Pro Chat",
            )

    upload_btn.click(process_files_for_rag, inputs=[files], outputs=[status])

demo.launch(share=True)

🧠 Model Load ho raha hai... (PyTorch 2.5.1 Stable Mode)


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [6]:
# ==========================================
# 🚀 TOOLS (RAG + WEB SEARCH)
# ==========================================
vector_db = None
search_tool = DuckDuckGoSearchRun()

def process_files_for_rag(files):
    global vector_db
    if not files: return "⚠️ Koi file nahi mili."

    documents = []
    status = ""
    for file in files:
        try:
            if file.name.endswith(".pdf"):
                loader = PyPDFLoader(file.name)
                documents.extend(loader.load())
                status += f"📄 PDF: {os.path.basename(file.name)}\n"
            elif file.name.endswith(".txt"):
                loader = TextLoader(file.name)
                documents.extend(loader.load())
                status += f"📝 TXT: {os.path.basename(file.name)}\n"
        except Exception as e:
            status += f"❌ Error: {file.name} - {str(e)}\n"

    if documents:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = text_splitter.split_documents(documents)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vector_db = Chroma.from_documents(chunks, embeddings)
        return status + f"\n✅ Database Ready! ({len(chunks)} chunks)"
    return "❌ Koi text extract nahi hua."

# ==========================================
# 🚀 SMART CHAT LOGIC
# ==========================================
def smart_chat(prompt, history, use_web=False):
    global vector_db

    context_text = ""
    system_instruction = "You are an expert Developer & AI Assistant. Answer in Roman Urdu mixed with English."

    # 1. RAG Check
    if vector_db:
        docs = vector_db.similarity_search(prompt, k=3)
        rag_content = "\n".join([d.page_content for d in docs])
        context_text += f"\n\n[📂 File Context]:\n{rag_content}"
        system_instruction += " Use the File Context to answer."

    # 2. Web Search Check
    if use_web:
        try:
            print("🌐 Searching Internet...")
            web_results = search_tool.run(prompt)
            context_text += f"\n\n[🌍 Web Search Results]:\n{web_results}"
            system_instruction += " Use the Web Search Results for latest info."
        except Exception as e:
            context_text += f"\n[Web Error]: {str(e)}"

    # 3. Prompt Construction
    messages = [{"role": "system", "content": system_instruction}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    final_user_message = f"{prompt}\n{context_text}"
    messages.append({"role": "user", "content": final_user_message})

    # 4. Generate
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        temperature=0.7,
        do_sample=True
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# ==========================================
# 🚀 API SETUP (FastAPI)
# ==========================================
app = FastAPI()

class APIRequest(BaseModel):
    prompt: str
    use_web: bool = False

@app.post("/chat")
def api_endpoint(req: APIRequest):
    response = smart_chat(req.prompt, [], req.use_web)
    return {"response": response}

def run_api():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_api)
thread.start()

try:
    public_url = ngrok.connect(8000).public_url
    print(f"\n🌍 API LIVE: {public_url}")
except:
    print("⚠️ Ngrok Error. Token check karein.")

# ==========================================
# 🚀 UI SETUP (Gradio)
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(), title="Ultimate AI Studio") as demo:
    gr.Markdown("# 🚀 Ultimate AI Studio (Chat + Files + Internet + API)")
    gr.Markdown(f"**API Endpoint:** `{public_url}/chat`")

    with gr.Row():
        with gr.Column(scale=1, variant="panel"):
            gr.Markdown("### 📂 Upload Knowledge")
            files = gr.File(file_count="multiple", label="PDF/TXT")
            upload_btn = gr.Button("Process Files", variant="primary")
            status = gr.Textbox(label="Status", interactive=False)

            gr.Markdown("### 🌐 Tools")
            web_checkbox = gr.Checkbox(label="Enable Internet Search", value=False)

        with gr.Column(scale=3):
            chat = gr.ChatInterface(
                fn=smart_chat,
                additional_inputs=[web_checkbox],
                title="Qwen Pro Chat",
            )

    upload_btn.click(process_files_for_rag, inputs=[files], outputs=[status])

demo.launch(share=True)

INFO:     Started server process [7384]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
/tmp/ipython-input-549660533.py:115: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Ultimate AI Studio") as demo:



🌍 API LIVE: https://<your-ngrok-url>
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://<your-gradio-share-link>.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading
import gradio as gr
import os

# RAG & Search Imports
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.tools import DuckDuckGoSearchRun

# ==========================================
# 🔑 APNA NGROK TOKEN YAHAN PASTE KAREIN
# ==========================================
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"
# ==========================================

# Token Setup
try:
    ngrok.kill() # Purane tunnels band karein
    if NGROK_AUTH_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE":
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
except:
    pass

# ==========================================
# 🚀 LOAD MODEL (Agar pehle se loaded hai to skip karega)
# ==========================================
try:
    if model: print("✅ Model already loaded.")
except NameError:
    print("🧠 Model Load ho raha hai...")
    model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_config,
        device_map="auto",
        torch_dtype=torch.float16
    )

# ==========================================
# 🚀 TOOLS
# ==========================================
vector_db = None
search_tool = DuckDuckGoSearchRun()

def process_files_for_rag(files):
    global vector_db
    if not files: return "⚠️ Koi file nahi mili."

    # Memory Cleanup before processing
    gc.collect()
    torch.cuda.empty_cache()

    documents = []
    status = ""
    for file in files:
        try:
            if file.name.endswith(".pdf"):
                loader = PyPDFLoader(file.name)
                documents.extend(loader.load())
                status += f"📄 PDF: {os.path.basename(file.name)}\n"
            elif file.name.endswith(".txt"):
                loader = TextLoader(file.name)
                documents.extend(loader.load())
                status += f"📝 TXT: {os.path.basename(file.name)}\n"
        except Exception as e:
            status += f"❌ Error: {file.name} - {str(e)}\n"

    if documents:
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        chunks = text_splitter.split_documents(documents)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        vector_db = Chroma.from_documents(chunks, embeddings)
        return status + f"\n✅ Database Ready! ({len(chunks)} chunks)"
    return "❌ Koi text extract nahi hua."

# ==========================================
# 🚀 SMART CHAT LOGIC (Anti-Crash Version)
# ==========================================
def smart_chat(prompt, history, use_web=False):
    # 1. Memory Cleanup (Sab se zaroori step)
    gc.collect()
    torch.cuda.empty_cache()

    global vector_db
    context_text = ""
    system_instruction = "You are an expert Developer. Answer concisely in Roman Urdu/English."

    # RAG
    if vector_db:
        try:
            docs = vector_db.similarity_search(prompt, k=2) # Sirf top 2 results layenge taake memory bache
            rag_content = "\n".join([d.page_content[:500] for d in docs]) # Content ko trim kiya
            context_text += f"\n\n[File Context]:\n{rag_content}"
        except: pass

    # Web Search
    if use_web:
        try:
            web_results = search_tool.run(prompt)
            context_text += f"\n\n[Web Search]:\n{web_results[:1000]}" # Sirf pehle 1000 characters
        except: pass

    # Message Construction (Limit History)
    messages = [{"role": "system", "content": system_instruction}]

    # Sirf pichli 2 conversations yaad rakhein (Memory bachane ke liye)
    for user_msg, bot_msg in history[-2:]:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    final_user_message = f"{prompt}\n{context_text}"
    messages.append({"role": "user", "content": final_user_message})

    # Generate
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    # Generation Config (Thora light rakha hai)
    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=768, # Limit kam ki taake crash na ho
        temperature=0.7,
        do_sample=True
    )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Final Cleanup
    del model_inputs, generated_ids
    torch.cuda.empty_cache()

    return response

# ==========================================
# 🚀 API & UI (Stable Launch)
# ==========================================
app = FastAPI()
class APIRequest(BaseModel):
    prompt: str
    use_web: bool = False

@app.post("/chat")
def api_endpoint(req: APIRequest):
    return {"response": smart_chat(req.prompt, [], req.use_web)}

def run_api():
    nest_asyncio.apply()
    try:
        uvicorn.run(app, host="0.0.0.0", port=8000)
    except: pass

thread = threading.Thread(target=run_api, daemon=True)
thread.start()

try:
    public_url = ngrok.connect(8000).public_url
    print(f"\n🌍 API LIVE: {public_url}")
except:
    print("⚠️ Ngrok Error")

# UI Setup with Queue (Crash Prevention)
with gr.Blocks(theme=gr.themes.Soft(), title="Ultimate AI Lite") as demo:
    gr.Markdown(f"# 🚀 AI Studio (Stable Version)\nAPI: `{public_url}/chat`")

    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(file_count="multiple", label="PDF/TXT")
            upload_btn = gr.Button("Process Files")
            status = gr.Textbox(label="Status", interactive=False)
            web_check = gr.Checkbox(label="Internet Search", value=False)
        with gr.Column(scale=3):
            chat = gr.ChatInterface(fn=smart_chat, additional_inputs=[web_check])

    upload_btn.click(process_files_for_rag, inputs=[files], outputs=[status])

# QUEUE ENABLED (Yeh crash rokega)
demo.queue().launch(share=True, debug=True)

🧠 Model Load ho raha hai...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO:     Started server process [657]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🌍 API LIVE: https://<your-ngrok-url>


/tmp/ipython-input-3135958648.py:180: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Ultimate AI Lite") as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://<your-gradio-share-link>.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
/tmp/ipython-input-3135958648.py:88: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [8]:
import requests

# Aapka Naya API Link
API_URL = "https://<your-ngrok-url>"

print("🤖 Connecting to Colab Brain...")

# 1. Simple Chat
payload = {
    "prompt": "What is the current price of Bitcoin in USD today?",
    "use_web": True
}

try:
    response = requests.post(f"{API_URL}/chat", json=payload)
    if response.status_code == 200:
        print("\n✅ AI Response:\n")
        print(response.json()["response"])
    else:
        print("❌ Error:", response.text)

except Exception as e:
    print("Connection Failed:", e)

🤖 Connecting to Colab Brain...
🌐 Searching Internet...

✅ AI Response:

The current price of Bitcoin in USD today is $68,922.73.
